# Train YOLOv8 known-defect detector on Kaggle (NEU-DET)

Clones the actual repo (private) using a GitHub token stored as a
Kaggle Secret, then runs the same `preprocessing/voc_to_yolo.py` and
`training/train.py` scripts used locally — no duplicated logic here.

Before running:

1. **Add-ons > Secrets**: make sure `GITHUB_TOKEN` is checked ON for
   this notebook (a secret being *saved* in your account isn't enough —
   it must be attached per-notebook). Ideally this token is scoped to
   just this one private repo with read-only contents access, not a
   broad `repo` scope, since it'll be live in this notebook's session.
2. **Add Input**: attach the
   [NEU Surface Defect Database](https://www.kaggle.com/datasets/kaustubhdikshit/neu-surface-defect-database)
   dataset.
3. **Settings > Accelerator**: GPU (T4 x2 or P100).
4. **Settings > Internet**: On.

Then run all cells top to bottom.

## 1. Clone the repo using the GitHub token secret

The token never gets printed, and the clone URL (which briefly embeds
it) is immediately overwritten in `.git/config` right after cloning —
so nothing sensitive lingers in `/kaggle/working` if this notebook's
output is ever saved or shared.

In [ ]:
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_OWNER_REPO = 'satyazm/factory-defect-detection'
CLONE_DIR = '/kaggle/working/repo'

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
auth_url = f'https://{token}@github.com/{GITHUB_OWNER_REPO}.git'
clean_url = f'https://github.com/{GITHUB_OWNER_REPO}.git'

result = subprocess.run(
    ['git', 'clone', '--depth', '1', auth_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    sanitized = result.stderr.replace(token, '***')  # never let the raw token hit notebook output
    raise RuntimeError(f'git clone failed:\n{sanitized}')

# Overwrite the stored remote URL so the token doesn't persist in .git/config
subprocess.run(['git', '-C', CLONE_DIR, 'remote', 'set-url', 'origin', clean_url], check=True)
del token, auth_url  # drop references now that we're done with them

print('Cloned to', CLONE_DIR)


In [ ]:
%cd /kaggle/working/repo


## 2. Install dependencies

Only what `training/train.py` needs — skips the heavier `anomalib`/`torch`-for-CPU pin from `requirements.txt`, since Kaggle's GPU image already ships a CUDA-enabled torch.

In [ ]:
!pip install -q ultralytics


## 3. Locate the attached dataset and convert VOC XML to YOLO format

Kaggle's exact mount path/nesting can vary by how the dataset was
packaged, so this inspects `/kaggle/input` first rather than assuming
a hardcoded path.

In [ ]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    depth = root[len('/kaggle/input'):].count(os.sep)
    if depth >= 3:
        dirs[:] = []
        continue
    print(root)


In [ ]:
import glob

candidates = [
    c for c in glob.glob('/kaggle/input/**/train', recursive=True)
    if os.path.isdir(os.path.join(c, 'images')) and os.path.isdir(os.path.join(c, 'annotations'))
]
assert candidates, (
    'Could not auto-locate the NEU-DET train/ folder under /kaggle/input — '
    'check the directory listing printed above and set NEU_DET_ROOT manually.'
)
NEU_DET_ROOT = os.path.dirname(candidates[0])
print('Detected NEU-DET root:', NEU_DET_ROOT)


In [ ]:
!python preprocessing/voc_to_yolo.py --source "{NEU_DET_ROOT}"


## 4. Train

Runs the repo's own `training/train.py` unmodified — same script you'd
run locally, just with a GPU underneath it this time.

In [ ]:
!python training/train.py --model yolov8n.pt --epochs 100 --imgsz 640


## 5. Evaluate (optional)

In [ ]:
!python training/evaluate.py --weights models/yolov8/defect_detector/weights/best.pt --split test


## 6. Get the trained weights back to your local repo

Zip it below, then use the notebook's **Output** pane (after *Save
Version → Save & Run All*) to download it. Copy the extracted `best.pt`
into your local repo at `models/yolov8/defect_detector/weights/best.pt`
— it's gitignored, so this is a manual file copy, not a git operation.

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/best_weights', 'zip', '/kaggle/working/repo/models/yolov8/defect_detector/weights')
print('Zipped weights at /kaggle/working/best_weights.zip')
